In [22]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from pathlib import Path
import numpy as np
import glob
import os

In [23]:
data_pipeline = "impute"
input_pipeline = "ohe"

In [24]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [25]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [26]:
cat_cols = ["gender"]
ordinal_cols = ["stress_level"]
binary_cols = ["academic_work_impact"]

In [27]:
ohe_dir = os.path.join(data_path, input_pipeline)
train_files = glob.glob(os.path.join(ohe_dir, "train_*.parq"))
test_files = glob.glob(os.path.join(ohe_dir, "test_*.parq"))
y = pd.read_csv(Path(ohe_dir) / '../raw/train.csv')[target_column]

In [28]:
for X_file, X_test_file in zip(train_files, test_files):
    X = ParquetFile(X_file).to_pandas()
    X_test = ParquetFile(X_test_file).to_pandas()

    bin_cols = binary_cols + (X.select_dtypes(include="bool").columns.to_list())
    X_cv_imputed = np.empty((X.shape[0], X.shape[1]), dtype=float)

    kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

    for train_index, valid_index in kf.split(X, y):
        X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
        y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
        
        imp = SimpleImputer(strategy='median')
        
        X_train_imp = imp.fit_transform(X_train)
        X_valid_imp = imp.transform(X_valid)
        
        X_cv_imputed[valid_index] = X_valid_imp

    X_cv_imputed = pd.DataFrame(X_cv_imputed, columns=X.columns, index=X.index)

    imp = SimpleImputer(strategy='median')
    imp.fit(X)
    X_test_imp = imp.transform(X_test)
    X_test_cv_imputed = pd.DataFrame(X_test_imp, columns=X_test.columns, index=X_test.index)

    X = X_cv_imputed.copy()
    X_test = X_test_cv_imputed.copy()

    for col in bin_cols:
        if col not in X:
            continue
        X[col] = X[col].round().clip(0, 1).astype(bool)
        X_test[col] = X_test[col].round().clip(0, 1).astype(bool)

    out_path = Path(data_path) / f"{data_pipeline}"
    out_path.mkdir(parents=True, exist_ok=True)

    write(out_path / Path(X_file).name, X)
    write(out_path / Path(X_test_file).name, X_test)